In [3]:
import numpy as np
import matplotlib.pyplot as plt

# Project Proposal: Simulation of a Modular BBM92 Receiver Subsystem

To ensure modularity and ease of integration with external tools, the system is designed as a pipeline of independent processing blocks. The data flow strictly separates the **"Quantum Source"** (Generation), the **"Channel"** (Transmission), and the **"Receiver"** (Detection and Processing), allowing for individual components to be swapped or upgraded without disrupting the system logic.

### Module 1: Input Source Module
This module creates the ground-truth data required to validate the receiver.

* **Function:** `gen_entangled_pairs(N)`
    * **Input:** Simulation Parameters (Number of pairs $N$, Entanglement Type).
    * **Logic:** Generates $N$ object pairs. To simulate entanglement logic in a classical environment, each object is assigned pre-correlated hidden variables for both Rectilinear (Z) and Diagonal (X) bases (e.g., if Alice has $Z=0$, Bob is pre-set to $Z=1$ for $\Psi^-$).
    * **Output:** A list of $N$ `QuantumPair` objects.

* **Function:** `assign_timestamps()`
    * **Input:** The generated `QuantumPair` list.
    * **Logic:** Assigns an initial emission time $t=0$ (or a scheduled sequence) to both photons in a pair, ensuring perfect synchronization ($t_A = t_B$) before channel effects are applied.
    * **Output:** Synchronized, timestamped photon list.

In [14]:
## POSSIBLY ADD ENTANGLEMENT TYPE PARAMETER LATER
class Source:
    def __init__(self):
        self.n_pairs = n_pairs
    def gen_entangled_pairs(self):
        # Empty dict to hold pairs
        print("Generating", self.n_pairs, "entangled pairs...")
        pairs = []
        for i in range(self.n_pairs):
            # Create a QuantumPair with pre-correlated hidden variables
            pair = QuantumPair(pair_id=i)
            pairs.append(pair)
        return pairs

    def assign_timestamps(self, pairs, separation_ns=10, sequential_mode=True):

        # Useful for debugging with no time separation
        if sequential_mode == False:
            print("Assigning no time separation for debugging...")
            for pair in pairs:
                # Assign synchronized timestamps
                pair.timestamp_A = 0
                pair.timestamp_B = 0
        else:
            print("Assigning time separation of", separation_ns, "...")
            for pair in pairs:
                # Assign staggered timestamps
                pair.timestamp_A = pair.pair_id * separation_ns
                pair.timestamp_B = pair.pair_id * separation_ns
        return pairs

class QuantumPair:
    def __init__(self, pair_id):
        self.pair_id = pair_id
        # Randomly assign hidden variables for Z and X bases
        self.alice_Z = np.random.choice([0, 1])
        self.bob_Z = 1 - self.alice_Z  # Anti-correlated for Psi-
        self.alice_X = np.random.choice([0, 1])
        self.bob_X = 1 - self.alice_X  # Anti-correlated for Psi-
        self.timestamp_A = 0
        self.timestamp_B = 0

    def __repr__(self):
        # Print assignments
        return (f"Pair {self.pair_id}: "
                f"Alice(Z={self.alice_Z}, X={self.alice_X}, t={self.timestamp_A}), "
                f"Bob(Z={self.bob_Z}, X={self.bob_X}, t={self.timestamp_B})")

In [17]:
# Test the Source module
n_pairs = 100
source = Source()
pairs = source.gen_entangled_pairs()
timestamped_pairs = source.assign_timestamps(pairs, sequential_mode=False)
for pair in timestamped_pairs:
    print(pair)

Generating 100 entangled pairs...
Assigning no time separation for debugging...
Pair 0: Alice(Z=0, X=0, t=0), Bob(Z=1, X=1, t=0)
Pair 1: Alice(Z=0, X=1, t=0), Bob(Z=1, X=0, t=0)
Pair 2: Alice(Z=1, X=0, t=0), Bob(Z=0, X=1, t=0)
Pair 3: Alice(Z=1, X=1, t=0), Bob(Z=0, X=0, t=0)
Pair 4: Alice(Z=0, X=1, t=0), Bob(Z=1, X=0, t=0)
Pair 5: Alice(Z=1, X=1, t=0), Bob(Z=0, X=0, t=0)
Pair 6: Alice(Z=1, X=0, t=0), Bob(Z=0, X=1, t=0)
Pair 7: Alice(Z=0, X=1, t=0), Bob(Z=1, X=0, t=0)
Pair 8: Alice(Z=0, X=1, t=0), Bob(Z=1, X=0, t=0)
Pair 9: Alice(Z=1, X=1, t=0), Bob(Z=0, X=0, t=0)
Pair 10: Alice(Z=0, X=0, t=0), Bob(Z=1, X=1, t=0)
Pair 11: Alice(Z=0, X=0, t=0), Bob(Z=1, X=1, t=0)
Pair 12: Alice(Z=1, X=1, t=0), Bob(Z=0, X=0, t=0)
Pair 13: Alice(Z=1, X=0, t=0), Bob(Z=0, X=1, t=0)
Pair 14: Alice(Z=0, X=0, t=0), Bob(Z=1, X=1, t=0)
Pair 15: Alice(Z=0, X=0, t=0), Bob(Z=1, X=1, t=0)
Pair 16: Alice(Z=0, X=0, t=0), Bob(Z=1, X=1, t=0)
Pair 17: Alice(Z=1, X=1, t=0), Bob(Z=0, X=0, t=0)
Pair 18: Alice(Z=0, X=0, t=0),

### Module 2: Channel Interface (Mock Channel)
This module degrades the perfect signal to mimic satellite downlink conditions.

* **Function:** `apply_geometric_loss(dB)`
    * **Input:** Perfect photon list and a loss parameter (e.g., 40-60 dB).
    * **Logic:** Simulates diffraction-limited beam spreading. The function stochastically deletes photons from the list with a probability $P = 1 - 10^{-dB/10}$.
    * **Output:** A sparse list of surviving photons.

* **Function:** `add_turbulence_jitter(sigma)`
    * **Input:** Sparse photon list and sigma.
    * **Logic:** Adds random Gaussian noise to the arrival timestamps of surviving photons ($t_{final} = t_{initial} + \mathcal{N}(0, \sigma)$). This simulates path-length fluctuations caused by atmospheric turbulence.
    * **Output:** Jittered photon list.

### Module 3: Receiver Subsystem (Detector Hardware)
This module simulates the physical response of the optical ground station.

* **Function:** `efficiency_filter(eta)`
    * **Input:** Jittered photon list and efficiency $\eta$.
    * **Logic:** Simulates detector inefficiency. A photon is only registered if a random variable check exceeds the detector efficiency threshold ($\eta \approx 60\%$ for Si-APDs).
    * **Output:** Detected photon list.

* **Function:** `inject_dark_counts(Hz)`
    * **Input:** Simulation duration and rate parameter.
    * **Logic:** Injects false detection events into the data stream based on Poissonian statistics. The rate is set to $\approx 100$ Hz to model SNSPD performance.
    * **Output:** Noisy photon list (Signal + Noise).

* **Function:** `measure_basis()`
    * **Input:** The noisy photon list.
    * **Logic:** Simulates passive basis selection. Each event is randomly assigned to the Z or X basis (50/50 probability).
    * **Output:** Two independent Detection Logs (Alice and Bob) containing: `[Timestamp, Basis, Bit_Value]`.

### Module 4: Receiver Subsystem (Processing Logic)
This component performs the classical algorithms required to extract the key.

* **Function:** `find_coincidences(tau_c)`
    * **Input:** The two independent Detection Logs.
    * **Logic:** Scans both logs to identify pairs where $|t_A - t_B| < \tau_c$. The window $\tau_c$ (typically 1-2 ns) is critical for distinguishing signal from noise.
    * **Output:** List of Raw Coincidences.

* **Function:** `sift_keys()`
    * **Input:** List of Raw Coincidences.
    * **Logic:** Compares the basis choice of Alice and Bob for each coincidence. Discards events where bases are incompatible.
    * **Output:** Sifted Key.

* **Function:** `calc_QBER()`
    * **Input:** The Sifted Key.
    * **Logic:** Compares the bit values of the sifted key. Calculates the ratio of errors to total bits.
    * **Output:** Final QBER percentage and Secure Key Rate.